In [1]:
import os
from pathlib import Path

os.chdir("..")
Path.cwd()

PosixPath('/home/emmanuel/Desktop/research_peft')

### Implementing the tokenizer

Things to do:
- load the tokenizer    (Done)
- load model            (Done)
- Download dataset 
- Preprocess dataset
- Tokenize dataset
- Create DataLoader
- Configure Trainer
- Train model
- Save checkpoint
- Load checkpoint
- Evaluate checkpoint

In [ ]:
from dataclasses import dataclass


@dataclass
class ModelConfig:
    model_name_or_path: str = "Qwen/Qwen2.5-1.5B"
    tokenizer_name: str = None
    trust_remote_code: bool = True

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


# loading the tokenizer
def load_tokenizer(model_cfg: ModelConfig):
    name = model_cfg.tokenizer_name or model_cfg.model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(
        name,
        trust_remote_code=model_cfg.trust_remote_code,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

In [ ]:
from typing import Any

import torch


def load_base_model(model_cfg: ModelConfig):
    """Load base model"""

    model_kwargs: dict[str, Any] = {
        "pretrained_model_name_or_path": model_cfg.model_name_or_path,
        "trust_remote_code": model_cfg.trust_remote_code,
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    }

    if model_cfg.cache_dir:
        model_kwargs["cache_dir"] = model_cfg.cache_dir
    if model_cfg.use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    return AutoModelForCausalLM.from_pretrained(**model_kwargs)

In [ ]:
@dataclass
class DataConfig:
    dataset_name: str | None = None
    dataset_config: str | None = None
    train_file: str | None = None
    val_file: str | None = None
    text_column: str = "text"
    prompt_column: str | None = "instruction"
    response_column: str | None = "output"
    max_seq_length: int = 2048
    instruction_template: str = "### Instruction:\n"
    response_template: str = "### Response:\n"
    val_split: float = 0.05
    num_proc: int = 4

Todo:
- download dataset
- process and prepare dataset
- save dataset
- tokenize dataset

In [9]:
!uv pip install datasets

Checked 1 package in 3.30s


In [3]:
from datasets import load_dataset

ds = load_dataset("yahma/alpaca-cleaned")

Generating train split: 100%|██████████| 51760/51760 [00:03<00:00, 14115.22 examples/s]


In [ ]:
def download_and_process_dataset():
    pass

In [ ]:
from datasets import load_dataset

data = "yahma/alpaca-cleaned"
ds = load_dataset(data_dir="dataset/alpaca_data_cleaned.json")
# save to /home/emmanuel/Desktop/research_peft/datasets
# ds.save_to_disk("/home/emmanuel/Desktop/research_peft/dataset/alpaca-cleaned")

In [ ]:
import pandas as pd

df = pd.read_json("hf://datasets/yahma/alpaca-cleaned/alpaca_data_cleaned.json")
# save to /home/emmanuel/Desktop/research_peft/datasets

df.to_json(
    "/home/emmanuel/Desktop/research_peft/dataset/alpaca_data_cleaned.json",
    orient="records",
    indent=2,
)

In [ ]:
from scripts.trainer import load_and_prepare_data

data = "dataset/alpaca_data_cleaned.json"

data_config = DataConfig(train_file=data)
train_data, val_data = load_and_prepare_data(data_config)

TypeError: 'NoneType' object is not iterable

In [ ]:
# Dataset Preparation
def formt_instruction(sample: dict, data_cfg: DataConfig) -> str:
    """Format dataset into instruction-following format."""
    instruct = []
    for item in sample:
        instruction = item.get(data_cfg.prompt_column, "")
        context = item.get("input", "")
        response = item.get(data_cfg.response_column, "")

        if context:
            instruct.append(
                f"{data_cfg.instruction_template}{instruction}\n\n"
                f"### Context:\n{context}\n\n"
                f"{data_cfg.response_template}{response}"
            )
        instruct.append(
            f"{data_cfg.instruction_template}{instruction}\n\n{data_cfg.response_template}{response}"
        )

    # ds = Dataset.from_list(instruct)
    return instruct

In [43]:
# from dataset.download import ALPACA_PROMPT
ALPACA_PROMPT = (
    # "Below is an instruction that describes a task{context_str}. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "{input_block}"
    "### Response:\n{output}"
)

for i in val_data:
    [ALPACA_PROMPT.format((row) for row in i)]
    break

KeyError: 'instruction'

In [ ]:
import re
import string


def normalize_answer(text):
    """Standard normalization for QA evaluation."""
    text = text.lower()
    text = re.sub(r"^(a|an|the)\s+", "", text)  # Remove leading articles
    text = "".join(char for char in text if char not in set(string.punctuation))

    return " ".join(text.split())  # Normalize whitespace


def compute_exact_match_score(prediction, references):
    """Returns 1.0 if normalized prediction matches any reference, else 0.0."""
    normalized_pred = normalize_answer(prediction)
    normalized_refs = [normalize_answer(ref) for ref in references]
    return float(normalized_pred in normalized_refs)

In [ ]:
import json
import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from datasets import Dataset
from tqdm import tqdm

logger = logging.getLogger(__name__)


# Perplexity
@torch.inference_mode()
def compute_perplexity(
    model,
    tokenizer,
    texts: list[str],
    max_length: int = 1024,
    stride: int = 512,
) -> dict[str, float]:
    """
    Sliding-window perplexity (handles texts longer than context window).
    Returns mean PPL, median PPL, and standard deviation across the corpus.
    """
    model.eval()
    ppls = []

    for text in tqdm(texts, desc="Computing perplexity"):
        encodings = tokenizer(text, return_tensors="pt", truncation=False)
        input_ids = encodings.input_ids.to(model.device)
        seq_len = input_ids.size(1)

        nlls = []
        prev_end = 0

        for begin in range(0, seq_len, stride):
            end = min(begin + max_length, seq_len)
            target_len = end - prev_end
            input_chunk = input_ids[:, begin:end]
            target_chunk = input_chunk.clone()
            target_chunk[:, :-target_len] = -100  # mask previously seen tokens

            with torch.no_grad():
                outputs = model(input_chunk, labels=target_chunk)
                neg_ll = outputs.loss * target_len
            nlls.append(neg_ll)
            prev_end = end
            if end == seq_len:
                break

        ppl = torch.exp(torch.stack(nlls).sum() / prev_end).item()
        ppls.append(ppl)

    return {
        "perplexity_mean": round(float(np.mean(ppls)), 4),
        "perplexity_std": round(float(np.std(ppls)), 4),
        "perplexity_median": round(float(np.median(ppls)), 4),
    }


# Generation Quality Metrics
def compute_rouge(predictions: list[str], references: list[str]) -> dict[str, float]:
    """ROUGE-1, ROUGE-2, ROUGE-L scores."""
    try:
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        scores = {"rouge1": [], "rouge2": [], "rougeL": []}
        for pred, ref in zip(predictions, references):
            s = scorer.score(ref, pred)
            for k in scores:
                scores[k].append(s[k].fmeasure)
        return {k: round(float(np.mean(v)), 4) for k, v in scores.items()}

    except ImportError:
        logger.warning("Install rouge-score: pip install rouge-score")
        return {}


def compute_bertscore(
    predictions: list[str],
    references: list[str],
    lang: str = "en",
    model_type: str = "microsoft/deberta-xlarge-mnli",
) -> dict[str, float]:
    """BERTScore F1 (semantic similarity)."""
    try:
        from bert_score import score

        P, R, F1 = score(predictions, references, lang=lang, model_type=model_type, verbose=False)
        return {
            "bertscore_precision": round(P.mean().item(), 4),
            "bertscore_recall": round(R.mean().item(), 4),
            "bertscore_f1": round(F1.mean().item(), 4),
        }
    except ImportError:
        logger.warning("Install bert-score: pip install bert-score")
        return {}


def compute_bleu(prediction: list[str], reference: list[str]) -> dict[str, float]:

    import sacrebleu

    try:
        # Compute corpus BLEU
        res = []
        for pred, ref in zip(prediction, reference):
            result = sacrebleu.corpus_bleu(pred, ref)
            res.append(result.score)

        return {"Bleu Score": np.mean(res)}

    except ImportError:
        logger.warning("insall sacrebleu: pip install sacrebleu")
        return {}


# Instruction-Following Benchmark
@dataclass
class BenchmarkResult:
    model_name: str
    task: str
    metrics: dict[str, float] = field(default_factory=dict)
    num_samples: int = 0
    generation_config: dict = field(default_factory=dict)


def run_generation_benchmark(
    model,
    tokenizer,
    dataset: Dataset,
    prompt_template: str,
    reference_column: str = "output",
    max_new_tokens: int = 256,
    batch_size: int = 8,
    temperature: float = 0.1,
) -> BenchmarkResult:
    """Run full generation benchmark on a dataset split."""
    predictions, references = [], []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Evaluating"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = [prompt_template.format(**row) for row in batch]
        refs = [row[reference_column] for row in batch]

        ## encode input
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(model.device)
        # run model generation
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                pad_token_id=tokenizer.eos_token_id,
            )

        # decode the input
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Strip prompt from output
        for prompt, dec, ref in zip(prompts, decoded, refs):
            predictions.append(dec[len(prompt) :].strip())
            references.append(ref.strip())

    rouge = compute_rouge(predictions, references)
    bert = compute_bertscore(predictions, references)
    bleu = compute_bleu(predictions, references)

    return BenchmarkResult(
        model_name=getattr(model.config, "_name_or_path", "unknown"),
        task="generation",
        metrics={**rouge, **bert, **bleu},
        num_samples=len(dataset),
        generation_config={"max_new_tokens": max_new_tokens, "temperature": temperature},
    )


# ─────────────────────────────────────────────
# Comparison: Finetuned models
# ─────────────────────────────────────────────


def compare_models(
    base_model_path: str,
    adapter_path: str,
    eval_texts: list[str],
    prompt_template: str,
    output_json: str = "eval_comparison.json",
):
    """Side-by-side perplexity comparison of base vs fine-tuned model."""
    import LoraInference

    logger.info("Evaluating base model...")
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path, torch_dtype=torch.bfloat16, device_map="auto"
    )
    base_ppl = compute_perplexity(base_model, base_tokenizer, eval_texts)
    base_benchmark_result = run_generation_benchmark(
        base_model, base_tokenizer, eval_texts, prompt_template
    )
    base_result = base_benchmark_result.metrics

    logger.info("Evaluating fine-tuned model...")
    ft = LoraInference(base_model_path, adapter_path)
    ft_ppl = compute_perplexity(ft.model, ft.tokenizer, eval_texts)
    benchmark_result = run_generation_benchmark(ft.model, ft.tokenizer, eval_texts, prompt_template)
    ft_result = benchmark_result.metrics

    results = {
        "base_model": {"path": base_model_path, **base_ppl, **base_result},
        "finetuned_model": {"path": adapter_path, **ft_ppl, **ft_result},
        "delta": {k: round(base_ppl[k] - ft_ppl[k], 4) for k in base_ppl},
    }

    with Path.open(output_json, "w") as f:
        json.dump(results, f, indent=2)
    logger.info(f"Results saved to {output_json}")
    return results

In [ ]:
from scripts.inference import load_model_with_adapter


def benchmark_result(
    base_model_path: str,
    adapter_path: list[str],
    eval_texts: list[str],
    prompt_template: str,
    output_json: str = "eval_comparison.json",
):
    """Side-by-side perplexity comparison of base vs fine-tuned model."""

    logger.info("Evaluating base model...")
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path, torch_dtype=torch.bfloat16, device_map="auto"
    )
    base_ppl = compute_perplexity(base_model, base_tokenizer, eval_texts)
    base_benchmark_result = run_generation_benchmark(
        base_model, base_tokenizer, eval_texts, prompt_template
    )
    base_result = base_benchmark_result.metrics

    logger.info("Evaluating fine-tuned models...")

    for adapter in adapter_path:
        ft = load_model_with_adapter(base_model_path, adapter)

        ft_ppl = compute_perplexity(ft.model, ft.tokenizer, eval_texts)
        benchmark_result = run_generation_benchmark(
            ft.model, ft.tokenizer, eval_texts, prompt_template
        )
        ft_result = benchmark_result.metrics

        results = {
            "base_model": {"path": base_model_path, **base_ppl, **base_result},
            "finetuned_model": {"path": adapter_path, **ft_ppl, **ft_result},
            "delta": {k: round(base_ppl[k] - ft_ppl[k], 4) for k in base_ppl},
        }

    with Path.open(output_json, "w") as f:
        json.dump(results, f, indent=2)
    logger.info(f"Results saved to {output_json}")
    return results

In [28]:
from datasets import load_dataset

ds = load_dataset(path="yahma/alpaca-cleaned")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yahma/alpaca-cleaned/12567cabf869d7c92e573c7c783905fc160e9639/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/12567cabf869d7c92e573c7c783905fc160e9639/alpaca-cleaned.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/yahma/alpaca-cleaned/yahma/alpaca-cleaned.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/12567cabf869d7c92e573c7c783905fc160e9639/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=yahma/alpaca-cleaned "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/a

In [29]:
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})

In [3]:
from scripts.dataset_utils import train_val_test_split

train_split = train_val_test_split(ds)

train_ds, val_ds, test_ds = train_split["train"], train_split["validation"], train_split["test"]

In [6]:
test_ds.save_to_disk("dataset/test_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/2588 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 2588/2588 [00:00<00:00, 8458.56 examples/s]


In [13]:
from datasets import load_from_disk

d = load_from_disk("dataset/test_dataset")

In [11]:
d

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 2588
})

In [ ]:
model_path = "outputs/checkpoints/qwen2.5-1.5b-finetune"

ModuleNotFoundError: No module named 'config'

In [14]:
from pathlib import Path

from datasets import load_dataset, load_from_disk

eval_text = "dataset/test_dataset"

ds = load_from_disk(eval_text)
# eval_texts = [json.loads(line) for line in Path(args.eval_texts).read_text().splitlines()]

In [2]:
def main(config_path: str):
    from scripts.config import DataConfig

    with Path.open(config_path) as f:
        cfg = json.load(f)

    data_cfg = DataConfig(**cfg.get("data", {}))

    if Path(f"dataset/{data_cfg.dataset_name}").is_dir():
        train_ds = load_from_disk(Path(data_cfg.train_file))
        val_ds = load_from_disk(Path(data_cfg.val_file))
        test_ds = load_from_disk(Path(data_cfg.test_file))
    else:
        train_ds, val_ds, test_ds = load_and_prepare_data(data_cfg)

        ## saving test set to disk for later evaluation
        data_cfg.test_file = f"dataset/{data_cfg.dataset_name}/test_file"
        data_cfg.train_file = f"dataset/{data_cfg.dataset_name}/train_file"
        data_cfg.val_file = f"dataset/{data_cfg.dataset_name}/val_file"

        test_ds.save_to_disk(data_cfg.test_file)
        train_ds.save_to_disk(data_cfg.train_file)
        val_ds.save_to_disk(data_cfg.val_file)
    print("done")

In [3]:
from scripts.trainer import load_and_prepare_data

main("scripts/config_file/config.json")

/home/emmanuel/Desktop/research_peft/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/emmanuel/Desktop/research_peft/scripts/evaluate.py:75: FutureWarning: Decorating classes is deprecated and will be disabled in future versions. You should only decorate functions or methods. To preserve the current behavior of class decoration, you can directly decorate the `__init__` method and nothing else.
  @torch.inference_mode()


Train: 35,197 | Val: 4,399 | Test: 4,400


Saving the dataset (1/1 shards): 100%|██████████| 4399/4399 [00:00<00:00, 69171.78 examples/s]

done


In [110]:
def format_prompt(data, data_cfg):
    prompts = []
    references = []

    for text in data:
        instruction = text["instruction"]
        input_d = text["input"]  # adjust field name to match your dataset schema
        reference = text["output"]  # or whatever the target/response field is called

        prompt = f"### Instruction:\n{instruction} {input_d}\n\n### Response:\n"
        prompts.append(prompt)
        references.append(reference)
    return prompt, reference

In [76]:
prompt, ref = format_prompt(test_ds)

TypeError: string indices must be integers, not 'str'

In [128]:
from pathlib import Path

from datasets import load_dataset, load_from_disk

from scripts.config import DataConfig


def eval_main(config_path: str):

    with Path.open(config_path) as f:
        cfg = json.load(f)

    data_cfg = DataConfig(**cfg.get("data", {}))

    if Path(f"dataset/{data_cfg.dataset_name}").is_dir():
        data_cfg.test_file = f"dataset/{data_cfg.dataset_name}/test_file"

        test_ds = load_from_disk(Path(data_cfg.test_file))

    # eval_text = {format_instruction(text, data_cfg) for text in test_ds}
    prompt_template = "{instruction} {input} \n\n### Response:\n"
    # prompt_template_with_context =  "{instruction} ### Context:\n{input}\n\n### Response: \n"

    prompts = [prompt_template.format(**row) for row in test_ds]
    # prompts = [prompt_template_with_context.format(**row)
    #          if row['input'] else
    #         prompt_template.format(**row)
    #        for row in test_ds
    # ]

    print(len(prompts))
    return prompts

In [129]:
prompts = eval_main("scripts/config_file/config.json")

4400


In [130]:
prompts[:20]

['Create a passphrase that contains five words with at least 10 characters without repeating any character.  \n\n### Response:\n',
 'Greet someone in Spanish  \n\n### Response:\n',
 'Describe the history of the automobile industry in the US.  \n\n### Response:\n',
 'Research the biography of a famous person and explain what has made them successful.  \n\n### Response:\n',
 'Identify three countries in South America  \n\n### Response:\n',
 'Provide an input to the following instruction: Write a story about a boy who travels to a magical world. Billy, a young outcast who loves to read fantasy books \n\n### Response:\n',
 'Guess what person is thinking by reading his/her following internal monologue. I wish I can be better at taking decisions, I feel that I make mistakes all the time. \n\n### Response:\n',
 'From the words given, form a sentence that conveys a feeling of hope and resilience. courage, sun, darkness \n\n### Response:\n',
 'Generate a list of five tasks that office workers s

In [115]:
prompts

'### Instruction:\nParse a given date into its correct components. String: 15/08/2022\n\n### Response:\n'

In [101]:
test_ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 4400
})

In [108]:
for text in test_ds:
    print(text)
    break

{'output': '"Discovering," "Innovative," "Sunflowers," "Abstaining," "Exploration."', 'input': '', 'instruction': 'Create a passphrase that contains five words with at least 10 characters without repeating any character.'}


In [107]:
test_ds["input"][:10]

['',
 '',
 '',
 '',
 '',
 'Billy, a young outcast who loves to read fantasy books',
 'I wish I can be better at taking decisions, I feel that I make mistakes all the time.',
 'courage, sun, darkness',
 '',
 '']

In [13]:
test_ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 4400
})

In [ ]:
from scripts.config import DataConfig
from scripts.trainer import format_instruction

data_cfg = DataConfig()
eval_text = [format_instruction(text, data_cfg) for text in test_ds]